In [1]:
# ============================================================
# LABELME -> YOLO
# Dataset structure:
#
# dataset_X/
# ├── images/
# │   ├── image_001.jpg
# │   └── ...
# └── labels/
#     ├── image_001.json
#     └── ...
#
# Output:
#
# dataset_X_yolo/
# ├── images/
# ├── labels/
# └── dataset.yaml
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import json
import shutil
import yaml
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_PATH = Path(
    r"Heavy Equipment_labelme"
)

OUTPUT_PATH = (
    DATASET_PATH.parent /
    f"{DATASET_PATH.name}_yolo"
)


# ------------------------------------------------------------
# Если классы известны заранее, можно указать их здесь.
#
# Например:
#
# CLASS_NAMES = [
#     "truck",
#     "excavator",
#     "concrete mixer"
# ]
#
# Если None — классы автоматически собираются из JSON.
# ------------------------------------------------------------

CLASS_NAMES = None


# ============================================================
# 2. PATHS
# ============================================================

IMAGES_DIR = DATASET_PATH / "images"
LABELS_DIR = DATASET_PATH / "labels"

IMAGES_OUT = OUTPUT_PATH / "images"
LABELS_OUT = OUTPUT_PATH / "labels"

IMAGES_OUT.mkdir(
    parents=True,
    exist_ok=True
)

LABELS_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. CHECK DIRECTORIES
# ============================================================

if not IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Не найдена папка images:\n{IMAGES_DIR}"
    )

if not LABELS_DIR.exists():
    raise FileNotFoundError(
        f"Не найдена папка labels:\n{LABELS_DIR}"
    )


print("=" * 70)
print("LABELME -> YOLO")
print("=" * 70)

print(f"Dataset: {DATASET_PATH}")
print(f"Output:  {OUTPUT_PATH}")
print()


# ============================================================
# 4. FIND FILES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff"
}


image_files = sorted([
    p
    for p in IMAGES_DIR.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )
])


json_files = sorted(
    LABELS_DIR.rglob("*.json")
)


print(f"Images: {len(image_files)}")
print(f"JSON:   {len(json_files)}")
print()


# ============================================================
# 5. COLLECT CLASSES
# ============================================================

detected_classes = set()

for json_path in json_files:

    try:

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

    except Exception:
        continue


    for shape in data.get(
        "shapes",
        []
    ):

        label = shape.get("label")

        if label:
            detected_classes.add(
                label.strip()
            )


# ------------------------------------------------------------
# CLASS LIST
# ------------------------------------------------------------

if CLASS_NAMES is None:

    CLASS_NAMES = sorted(
        detected_classes
    )

else:

    CLASS_NAMES = list(
        CLASS_NAMES
    )


CLASS_TO_ID = {
    name: idx
    for idx, name in enumerate(
        CLASS_NAMES
    )
}


print("Classes:")

for class_id, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"  {class_id}: {class_name}"
    )

print()


# ============================================================
# 6. IMAGE LOOKUP
# ============================================================

image_lookup = {}

for image_path in image_files:

    # Ключ — относительный путь без расширения
    relative = image_path.relative_to(
        IMAGES_DIR
    )

    key = str(
        relative.with_suffix("")
    ).replace(
        "\\",
        "/"
    )

    image_lookup[key] = image_path


# ============================================================
# 7. STATISTICS
# ============================================================

class_object_counts = Counter()

class_image_counts = defaultdict(set)

shape_type_counts = Counter()

objects_per_image = []

converted_files = []

missing_images = []
missing_json = []
invalid_json = []
unknown_classes = []
unsupported_shapes = []
empty_annotations = []


# ============================================================
# 8. SHAPE -> BBOX
# ============================================================

def shape_to_bbox(shape):

    shape_type = shape.get(
        "shape_type",
        "polygon"
    )

    points = shape.get(
        "points",
        []
    )


    # --------------------------------------------------------
    # RECTANGLE
    # --------------------------------------------------------

    if shape_type == "rectangle":

        if len(points) < 2:
            return None

        x1, y1 = points[0]
        x2, y2 = points[1]

        return (
            min(x1, x2),
            min(y1, y2),
            max(x1, x2),
            max(y1, y2)
        )


    # --------------------------------------------------------
    # POLYGON
    # --------------------------------------------------------

    if shape_type == "polygon":

        if len(points) < 3:
            return None

        xs = [
            point[0]
            for point in points
        ]

        ys = [
            point[1]
            for point in points
        ]

        return (
            min(xs),
            min(ys),
            max(xs),
            max(ys)
        )


    # --------------------------------------------------------
    # OTHER
    # --------------------------------------------------------

    return None


# ============================================================
# 9. BBOX -> YOLO
# ============================================================

def bbox_to_yolo(
    xmin,
    ymin,
    xmax,
    ymax,
    image_width,
    image_height
):

    # Ограничиваем координаты размерами изображения

    xmin = max(
        0,
        min(xmin, image_width)
    )

    xmax = max(
        0,
        min(xmax, image_width)
    )

    ymin = max(
        0,
        min(ymin, image_height)
    )

    ymax = max(
        0,
        min(ymax, image_height)
    )


    # Исправляем перевернутые координаты

    if xmax < xmin:
        xmin, xmax = xmax, xmin

    if ymax < ymin:
        ymin, ymax = ymax, ymin


    # YOLO center

    x_center = (
        (xmin + xmax) / 2
    ) / image_width

    y_center = (
        (ymin + ymax) / 2
    ) / image_height


    # YOLO width / height

    width = (
        xmax - xmin
    ) / image_width

    height = (
        ymax - ymin
    ) / image_height


    return (
        x_center,
        y_center,
        width,
        height
    )


# ============================================================
# 10. PROCESS JSON
# ============================================================

for json_path in json_files:

    # --------------------------------------------------------
    # Relative path
    # --------------------------------------------------------

    relative_json = json_path.relative_to(
        LABELS_DIR
    )

    relative_stem = str(
        relative_json.with_suffix("")
    ).replace(
        "\\",
        "/"
    )


    # --------------------------------------------------------
    # FIND IMAGE
    # --------------------------------------------------------

    image_path = image_lookup.get(
        relative_stem
    )


    if image_path is None:

        missing_images.append(
            str(relative_json)
        )

        continue


    # --------------------------------------------------------
    # READ JSON
    # --------------------------------------------------------

    try:

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

    except Exception as e:

        invalid_json.append({
            "json": str(relative_json),
            "error": str(e)
        })

        continue


    # --------------------------------------------------------
    # IMAGE SIZE
    # --------------------------------------------------------

    image_width = data.get(
        "imageWidth"
    )

    image_height = data.get(
        "imageHeight"
    )


    # Если размеров нет в JSON,
    # берём реальные размеры изображения.

    if (
        image_width is None
        or image_height is None
    ):

        from PIL import Image

        try:

            with Image.open(
                image_path
            ) as img:

                image_width, image_height = (
                    img.size
                )

        except Exception as e:

            invalid_json.append({
                "json": str(relative_json),
                "error": str(e)
            })

            continue


    image_width = float(
        image_width
    )

    image_height = float(
        image_height
    )


    # --------------------------------------------------------
    # SHAPES
    # --------------------------------------------------------

    shapes = data.get(
        "shapes",
        []
    )


    if len(shapes) == 0:

        empty_annotations.append(
            str(relative_json)
        )


    yolo_lines = []

    image_classes = set()


    for shape in shapes:

        label = shape.get(
            "label"
        )


        if not label:
            continue


        label = label.strip()


        # ----------------------------------------------------
        # CLASS
        # ----------------------------------------------------

        if label not in CLASS_TO_ID:

            unknown_classes.append({
                "json": str(relative_json),
                "class": label
            })

            continue


        class_id = CLASS_TO_ID[
            label
        ]


        # ----------------------------------------------------
        # SHAPE TYPE
        # ----------------------------------------------------

        shape_type = shape.get(
            "shape_type",
            "polygon"
        )

        shape_type_counts[
            shape_type
        ] += 1


        # ----------------------------------------------------
        # BBOX
        # ----------------------------------------------------

        bbox = shape_to_bbox(
            shape
        )


        if bbox is None:

            unsupported_shapes.append({
                "json": str(relative_json),
                "class": label,
                "shape_type": shape_type
            })

            continue


        xmin, ymin, xmax, ymax = bbox


        # ----------------------------------------------------
        # CONVERT
        # ----------------------------------------------------

        (
            x_center,
            y_center,
            box_width,
            box_height
        ) = bbox_to_yolo(
            xmin,
            ymin,
            xmax,
            ymax,
            image_width,
            image_height
        )


        # ----------------------------------------------------
        # INVALID BOX
        # ----------------------------------------------------

        if (
            box_width <= 0
            or box_height <= 0
        ):

            continue


        # ----------------------------------------------------
        # YOLO FORMAT
        # ----------------------------------------------------

        yolo_lines.append(
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}"
        )


        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        class_object_counts[
            class_id
        ] += 1

        image_classes.add(
            class_id
        )


    # --------------------------------------------------------
    # NOTHING TO SAVE
    # --------------------------------------------------------

    if len(yolo_lines) == 0:
        continue


    # ========================================================
    # OUTPUT PATH
    # ========================================================

    relative_image = image_path.relative_to(
        IMAGES_DIR
    )

    output_image_path = (
        IMAGES_OUT /
        relative_image
    )

    output_label_path = (
        LABELS_OUT /
        relative_image.with_suffix(".txt")
    )


    # Создаём вложенные папки,
    # если они есть в исходном датасете.

    output_image_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    output_label_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # COPY IMAGE
    # ========================================================

    shutil.copy2(
        image_path,
        output_image_path
    )


    # ========================================================
    # SAVE YOLO LABEL
    # ========================================================

    with open(
        output_label_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "\n".join(yolo_lines)
        )


    # ========================================================
    # IMAGE-LEVEL CLASS STATISTICS
    # ========================================================

    image_key = str(
        relative_image.with_suffix("")
    ).replace(
        "\\",
        "/"
    )


    for class_id in image_classes:

        class_image_counts[
            class_id
        ].add(
            image_key
        )


    objects_per_image.append(
        len(yolo_lines)
    )


    converted_files.append({
        "image": str(relative_image),
        "label": str(
            relative_image.with_suffix(
                ".txt"
            )
        ),
        "objects": len(yolo_lines)
    })


# ============================================================
# 11. FIND IMAGES WITHOUT JSON
# ============================================================

for image_path in image_files:

    relative_image = image_path.relative_to(
        IMAGES_DIR
    )

    relative_stem = str(
        relative_image.with_suffix("")
    ).replace(
        "\\",
        "/"
    )

    expected_json = (
        LABELS_DIR /
        relative_image.with_suffix(".json")
    )


    if not expected_json.exists():

        missing_json.append(
            str(relative_image)
        )


# ============================================================
# 12. CREATE YAML
# ============================================================

yaml_data = {
    "path": str(
        OUTPUT_PATH.resolve()
    ),

    "train": "images",

    "val": "images",

    "nc": len(CLASS_NAMES),

    "names": CLASS_NAMES
}


yaml_path = (
    OUTPUT_PATH /
    "dataset.yaml"
)


with open(
    yaml_path,
    "w",
    encoding="utf-8"
) as f:

    yaml.dump(
        yaml_data,
        f,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# 13. CLASS STATISTICS
# ============================================================

total_objects = sum(
    class_object_counts.values()
)

total_images = len(
    converted_files
)


class_rows = []


for class_id, class_name in enumerate(
    CLASS_NAMES
):

    objects = class_object_counts[
        class_id
    ]

    images = len(
        class_image_counts[
            class_id
        ]
    )


    object_percentage = (
        objects /
        total_objects *
        100
        if total_objects > 0
        else 0
    )


    image_percentage = (
        images /
        total_images *
        100
        if total_images > 0
        else 0
    )


    class_rows.append({

        "class_id": class_id,

        "class_name": class_name,

        "objects": objects,

        "object_percentage":
            object_percentage,

        "images": images,

        "image_percentage":
            image_percentage
    })


class_df = pd.DataFrame(
    class_rows
)


class_df.to_csv(
    OUTPUT_PATH /
    "conversion_class_statistics.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 14. SHAPE TYPE STATISTICS
# ============================================================

shape_df = pd.DataFrame(
    [
        {
            "shape_type": shape_type,
            "count": count
        }

        for shape_type, count
        in shape_type_counts.items()
    ]
)


shape_df.to_csv(
    OUTPUT_PATH /
    "shape_type_statistics.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 15. CONVERSION SUMMARY
# ============================================================

summary_df = pd.DataFrame([
    {
        "dataset": DATASET_PATH.name,

        "source_images":
            len(image_files),

        "source_json":
            len(json_files),

        "converted_images":
            total_images,

        "total_objects":
            total_objects,

        "classes":
            len(CLASS_NAMES),

        "images_without_json":
            len(missing_json),

        "json_without_images":
            len(missing_images),

        "invalid_json":
            len(invalid_json),

        "unknown_classes":
            len(unknown_classes),

        "unsupported_shapes":
            len(unsupported_shapes),

        "empty_annotations":
            len(empty_annotations),

        "mean_objects_per_image": (
            sum(objects_per_image) /
            len(objects_per_image)
            if objects_per_image
            else 0
        ),

        "min_objects_per_image": (
            min(objects_per_image)
            if objects_per_image
            else 0
        ),

        "max_objects_per_image": (
            max(objects_per_image)
            if objects_per_image
            else 0
        )
    }
])


summary_df.to_csv(
    OUTPUT_PATH /
    "conversion_summary.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 16. PROBLEM REPORTS
# ============================================================

if missing_json:

    pd.DataFrame({
        "image": missing_json
    }).to_csv(
        OUTPUT_PATH /
        "images_without_json.csv",

        index=False,

        encoding="utf-8-sig"
    )


if missing_images:

    pd.DataFrame({
        "json": missing_images
    }).to_csv(
        OUTPUT_PATH /
        "json_without_images.csv",

        index=False,

        encoding="utf-8-sig"
    )


if invalid_json:

    pd.DataFrame(
        invalid_json
    ).to_csv(
        OUTPUT_PATH /
        "invalid_json.csv",

        index=False,

        encoding="utf-8-sig"
    )


if unknown_classes:

    pd.DataFrame(
        unknown_classes
    ).drop_duplicates().to_csv(
        OUTPUT_PATH /
        "unknown_classes.csv",

        index=False,

        encoding="utf-8-sig"
    )


if unsupported_shapes:

    pd.DataFrame(
        unsupported_shapes
    ).to_csv(
        OUTPUT_PATH /
        "unsupported_shapes.csv",

        index=False,

        encoding="utf-8-sig"
    )


if empty_annotations:

    pd.DataFrame({
        "json": empty_annotations
    }).to_csv(
        OUTPUT_PATH /
        "empty_annotations.csv",

        index=False,

        encoding="utf-8-sig"
    )


# ============================================================
# 17. TEXT REPORT
# ============================================================

report = []

report.append(
    "LABELME -> YOLO CONVERSION REPORT"
)

report.append("=" * 70)

report.append(
    f"Source: {DATASET_PATH}"
)

report.append(
    f"Output: {OUTPUT_PATH}"
)

report.append("")

report.append(
    f"Source images: {len(image_files)}"
)

report.append(
    f"Source JSON: {len(json_files)}"
)

report.append(
    f"Converted images: {total_images}"
)

report.append(
    f"Total objects: {total_objects}"
)

report.append(
    f"Images without JSON: {len(missing_json)}"
)

report.append(
    f"JSON without image: {len(missing_images)}"
)

report.append(
    f"Invalid JSON: {len(invalid_json)}"
)

report.append(
    f"Unknown classes: {len(unknown_classes)}"
)

report.append(
    f"Unsupported shapes: {len(unsupported_shapes)}"
)

report.append(
    f"Empty annotations: {len(empty_annotations)}"
)

report.append("")

report.append(
    "CLASS DISTRIBUTION"
)

report.append("-" * 70)


for _, row in class_df.iterrows():

    report.append(
        f"{int(row['class_id']):3d} | "
        f"{row['class_name']:<30} | "
        f"objects: {int(row['objects']):7d} | "
        f"{row['object_percentage']:7.2f}% | "
        f"images: {int(row['images']):7d}"
    )


report.append("")

report.append(
    "SHAPE TYPES"
)

report.append("-" * 70)


for shape_type, count in (
    shape_type_counts.items()
):

    report.append(
        f"{shape_type}: {count}"
    )


with open(
    OUTPUT_PATH /
    "conversion_report.txt",

    "w",

    encoding="utf-8"
) as f:

    f.write(
        "\n".join(report)
    )


# ============================================================
# 18. FINAL
# ============================================================

print()
print("=" * 70)
print("CONVERSION FINISHED")
print("=" * 70)

print()

print(
    f"Images:            {len(image_files)}"
)

print(
    f"JSON:              {len(json_files)}"
)

print(
    f"Converted images:  {total_images}"
)

print(
    f"Objects:            {total_objects}"
)

print(
    f"Images without JSON:{len(missing_json)}"
)

print(
    f"JSON without image: {len(missing_images)}"
)

print(
    f"Invalid JSON:       {len(invalid_json)}"
)

print(
    f"Unknown classes:    {len(unknown_classes)}"
)

print(
    f"Unsupported shapes: {len(unsupported_shapes)}"
)

print()

print("Class distribution:")
print(
    class_df.to_string(index=False)
)

print()

print("Output:")
print(
    OUTPUT_PATH.resolve()
)

print()

print("YAML:")
print(
    yaml_path.resolve()
)

LABELME -> YOLO
Dataset: Heavy Equipment_labelme
Output:  Heavy Equipment_labelme_yolo

Images: 3882
JSON:   3882

Classes:
  0: bulldozer
  1: dump
  2: excavator
  3: roller


CONVERSION FINISHED

Images:            3882
JSON:              3882
Converted images:  3876
Objects:            6702
Images without JSON:0
JSON without image: 0
Invalid JSON:       0
Unknown classes:    0
Unsupported shapes: 0

Class distribution:
 class_id class_name  objects  object_percentage  images  image_percentage
        0  bulldozer     2043          30.483438    2034         52.476780
        1       dump     1835          27.379887    1563         40.325077
        2  excavator     2313          34.512086    2181         56.269350
        3     roller      511           7.624590     408         10.526316

Output:
C:\Users\Ева\Desktop\Хакатон\data_preproc\Heavy Equipment_labelme_yolo

YAML:
C:\Users\Ева\Desktop\Хакатон\data_preproc\Heavy Equipment_labelme_yolo\dataset.yaml
